In [9]:
import pandas as pd
import os
from collections import Counter
import json

# LOAD DATASET
FILE_PATH = r"D:\chiselon\Week 0\Week_0_Prep_Week_Ssample Data_clinic_cases.csv"

if not os.path.exists(FILE_PATH):
    raise FileNotFoundError("CSV file not found at given path.")

df = pd.read_csv(FILE_PATH)

print("Dataset Loaded Successfully!")
print("Total Cases:", len(df))


# TEXT PREPROCESSING
def preprocess_text(text):
    if not isinstance(text, str):
        return ""
    return text.lower().strip()


# Apply preprocessing to important columns
for col in ["Symptoms", "Diagnosis", "Treatment", "Outcome"]:
    if col in df.columns:
        df[col] = df[col].apply(preprocess_text)



# INSIGHT GENERATION
def generate_case_insight(similar_case_ids):


    # Extract similar cases
    similar_cases = df[df["case_id"].isin(similar_case_ids)]

    if similar_cases.empty:
        return {"message": "No similar cases found."}

    total_cases = len(similar_cases)

    # Aggregate fields
    all_symptoms = []
    all_treatments = []
    all_diagnoses = []
    recovered_count = 0
    recovery_days = []

    for _, row in similar_cases.iterrows():

        # Symptoms (split by comma)
        symptoms = row.get("Symptoms", "")
        all_symptoms.extend([s.strip() for s in symptoms.split(",") if s])

        # Treatments
        treatment = row.get("Treatment", "")
        if treatment:
            all_treatments.append(treatment)

        # Diagnosis
        diagnosis = row.get("Diagnosis", "")
        if diagnosis:
            all_diagnoses.append(diagnosis)

        # Outcome analysis
        outcome = row.get("Outcome", "")
        if "recover" in outcome:
            recovered_count += 1

        # recovery days 
        if "RecoveryDays" in df.columns:
            days = row.get("RecoveryDays")
            if pd.notnull(days):
                recovery_days.append(days)

    # Frequency Analysis
    symptom_freq = Counter(all_symptoms)
    treatment_freq = Counter(all_treatments)
    diagnosis_freq = Counter(all_diagnoses)

    top_symptoms = symptom_freq.most_common(5)
    top_treatments = treatment_freq.most_common(3)
    top_diagnosis = diagnosis_freq.most_common(1)

    # Recovery Trend
    recovery_rate = (recovered_count / total_cases) * 100

    avg_recovery_time = None
    if recovery_days:
        avg_recovery_time = sum(recovery_days) / len(recovery_days)

    # Clinical Interpretation
    if recovery_rate > 80:
        interpretation = "Most similar cases responded well to treatment."
    elif recovery_rate > 50:
        interpretation = "Moderate recovery observed. Monitoring advised."
    else:
        interpretation = "Low recovery rate. Consider alternative strategies."

    # Step 6: Structured Output
    insight_output = {
        "case_summary": {
            "total_similar_cases": total_cases,
            "dominant_diagnosis": top_diagnosis[0][0] if top_diagnosis else "N/A"
        },
        "symptom_patterns": {
            "top_symptoms": top_symptoms
        },
        "treatment_patterns": {
            "top_treatments": top_treatments
        },
        "recovery_trends": {
            "recovery_rate_percent": round(recovery_rate, 2),
            "average_recovery_days": round(avg_recovery_time, 2) if avg_recovery_time else "Not Available"
        },
        "clinical_interpretation": interpretation
    }

    return insight_output


#example testing 
example_similar_ids = df["case_id"].head(5).tolist()

insight = generate_case_insight(example_similar_ids)

print("\nGenerated Clinical Insight:\n")
print(json.dumps(insight, indent=4))

Dataset Loaded Successfully!
Total Cases: 10

Generated Clinical Insight:

{
    "case_summary": {
        "total_similar_cases": 5,
        "dominant_diagnosis": "N/A"
    },
    "symptom_patterns": {
        "top_symptoms": []
    },
    "treatment_patterns": {
        "top_treatments": []
    },
    "recovery_trends": {
        "recovery_rate_percent": 0.0,
        "average_recovery_days": "Not Available"
    },
    "clinical_interpretation": "Low recovery rate. Consider alternative strategies."
}


CASE INSIGHT GENERATION

In [72]:
from collections import Counter
import json

def analyze_new_patient(new_patient_symptoms, top_k=5):

    # Normalize column names
    df.columns = df.columns.str.strip().str.lower()

    # Ensure caseid exists
    if "caseid" not in df.columns:
        df["caseid"] = df.index.astype(str)

    # Clean new patient symptoms
    new_symptoms = [s.strip().lower() for s in new_patient_symptoms.split(",") if s.strip()]

    similarity_data = []

    # Compute Similarity 
    for _, row in df.iterrows():

        case_text = str(row["symptoms"]).lower()

        match_count = 0
        for symptom in new_symptoms:
            if symptom in case_text:
                match_count += 1

        score = match_count / len(new_symptoms) if len(new_symptoms) > 0 else 0

        similarity_data.append((row["caseid"], score))

    # Sort by similarity
    similarity_data = sorted(similarity_data, key=lambda x: x[1], reverse=True)

    # Take only top cases with similarity > 0
    top_cases = [case_id for case_id, score in similarity_data[:top_k] if score > 0]

    if not top_cases:
        return {"message": "No similar cases found in database."}

    similar_cases = df[df["caseid"].isin(top_cases)]


    # Insight Generation
    total_cases = len(similar_cases)

    all_symptoms = []
    all_treatments = []
    all_diagnoses = []
    recovered_count = 0

    for _, row in similar_cases.iterrows():

        all_symptoms.extend(
            [s.strip() for s in str(row["symptoms"]).split(",") if s.strip()]
        )

        all_treatments.append(str(row["treatment"]))
        all_diagnoses.append(str(row["diagnosis"]))

        # Improved recovery detection
        outcome_text = str(row["outcome"]).lower()

        if any(word in outcome_text for word in 
               ["recover", "improved", "resolved", "cleared", "successful"]):
            recovered_count += 1

    symptom_freq = Counter(all_symptoms)
    treatment_freq = Counter(all_treatments)
    diagnosis_freq = Counter(all_diagnoses)

    recovery_rate = (recovered_count / total_cases) * 100 if total_cases > 0 else 0

    # Interpretation Logic
    if recovery_rate >= 80:
        interpretation = (
            "Strong positive recovery trend observed among similar cases. "
            "Standard treatment protocols appear highly effective."
        )

    elif recovery_rate >= 50:
        interpretation = (
            "Moderate recovery trend observed. Most patients respond "
            "well with consistent treatment and monitoring."
        )

    elif recovery_rate >= 20:
        interpretation = (
            "Mixed clinical outcomes observed. Treatment response varies "
            "and closer follow-up is advised."
        )

    else:
        interpretation = (
            "Limited recovery observed in similar cases. Consider "
            "reassessment of diagnosis or alternative treatment strategies."
        )

    insight_output = {
        "new_patient_symptoms": new_patient_symptoms,
        "matched_cases_count": total_cases,
        "top_diagnosis": diagnosis_freq.most_common(1)[0][0] if diagnosis_freq else "N/A",
        "symptom_patterns": symptom_freq.most_common(5),
        "treatment_patterns": treatment_freq.most_common(3),
        "recovery_rate_percent": round(recovery_rate, 2),
        "clinical_summary": interpretation
    }

    return insight_output

TESTING THE INSIGHT GENERATION FOR SAMPLES (GIVEN ARTIFICIALLY)

In [75]:
patient_1 = "acne, oily skin"
result_1 = analyze_new_patient(patient_1)
print(json.dumps(result_1, indent=4))

{
    "new_patient_symptoms": "acne, oily skin",
    "matched_cases_count": 1,
    "top_diagnosis": "Acne Vulgaris",
    "symptom_patterns": [
        [
            "Acne on face",
            1
        ],
        [
            "oily skin",
            1
        ]
    ],
    "treatment_patterns": [
        [
            "Topical retinoid + benzoyl peroxide",
            1
        ]
    ],
    "recovery_rate_percent": 100.0,
    "clinical_summary": "Strong positive recovery trend observed among similar cases. Standard treatment protocols appear highly effective."
}


In [77]:
patient_2 = "Dark patches on cheeks"
result_2 = analyze_new_patient(patient_2)
print(json.dumps(result_2, indent=4))

{
    "new_patient_symptoms": "Dark patches on cheeks",
    "matched_cases_count": 1,
    "top_diagnosis": "Melasma",
    "symptom_patterns": [
        [
            "Dark patches on cheeks",
            1
        ]
    ],
    "treatment_patterns": [
        [
            "Chemical peel + sunscreen",
            1
        ]
    ],
    "recovery_rate_percent": 100.0,
    "clinical_summary": "Strong positive recovery trend observed among similar cases. Standard treatment protocols appear highly effective."
}


In [79]:
patient_3 = "hair fall, thinning"
result_3 = analyze_new_patient(patient_3)
print(json.dumps(result_3, indent=4))

{
    "new_patient_symptoms": "hair fall, thinning",
    "matched_cases_count": 1,
    "top_diagnosis": "Telogen Effluvium",
    "symptom_patterns": [
        [
            "Hair fall and thinning",
            1
        ]
    ],
    "treatment_patterns": [
        [
            "PRP therapy",
            1
        ]
    ],
    "recovery_rate_percent": 100.0,
    "clinical_summary": "Strong positive recovery trend observed among similar cases. Standard treatment protocols appear highly effective."
}


In [81]:
patient_4 = "red scaly patches, scalp"
result_4 = analyze_new_patient(patient_4)
print(json.dumps(result_4, indent=4))

{
    "new_patient_symptoms": "red scaly patches, scalp",
    "matched_cases_count": 1,
    "top_diagnosis": "Seborrheic Dermatitis",
    "symptom_patterns": [
        [
            "Red scaly patches on scalp",
            1
        ]
    ],
    "treatment_patterns": [
        [
            "Medicated shampoo",
            1
        ]
    ],
    "recovery_rate_percent": 100.0,
    "clinical_summary": "Strong positive recovery trend observed among similar cases. Standard treatment protocols appear highly effective."
}


In [83]:
patient_5 = "loose skin, wrinkles"
result_5 = analyze_new_patient(patient_5)
print(json.dumps(result_5, indent=4))

{
    "new_patient_symptoms": "loose skin, wrinkles",
    "matched_cases_count": 1,
    "top_diagnosis": "Skin Aging",
    "symptom_patterns": [
        [
            "Loose skin and wrinkles",
            1
        ]
    ],
    "treatment_patterns": [
        [
            "Botulinum toxin evaluation",
            1
        ]
    ],
    "recovery_rate_percent": 0.0,
    "clinical_summary": "Limited recovery observed in similar cases. Consider reassessment of diagnosis or alternative treatment strategies."
}


In [85]:
patient_1 = "acne, oily skin"
patient_2 = "dark patches"
patient_3 = "hair fall"
patient_4 = "red scaly patches"
patient_5 = "loose skin"

print(json.dumps(analyze_new_patient(patient_1), indent=4))
print("\n ------------------------------------------------------")
print(json.dumps(analyze_new_patient(patient_2), indent=4))
print("\n ------------------------------------------------------")
print(json.dumps(analyze_new_patient(patient_3), indent=4))
print("\n ------------------------------------------------------")
print(json.dumps(analyze_new_patient(patient_4), indent=4))
print("\n ------------------------------------------------------")
print(json.dumps(analyze_new_patient(patient_5), indent=4))

{
    "new_patient_symptoms": "acne, oily skin",
    "matched_cases_count": 1,
    "top_diagnosis": "Acne Vulgaris",
    "symptom_patterns": [
        [
            "Acne on face",
            1
        ],
        [
            "oily skin",
            1
        ]
    ],
    "treatment_patterns": [
        [
            "Topical retinoid + benzoyl peroxide",
            1
        ]
    ],
    "recovery_rate_percent": 100.0,
    "clinical_summary": "Strong positive recovery trend observed among similar cases. Standard treatment protocols appear highly effective."
}

 ------------------------------------------------------
{
    "new_patient_symptoms": "dark patches",
    "matched_cases_count": 1,
    "top_diagnosis": "Melasma",
    "symptom_patterns": [
        [
            "Dark patches on cheeks",
            1
        ]
    ],
    "treatment_patterns": [
        [
            "Chemical peel + sunscreen",
            1
        ]
    ],
    "recovery_rate_percent": 100.0,
    "clinical

INSIGHTS FROM THE PROCESS:

~> The similarity results are transformed into structured clinical insights.

~> The insights are aggregated by analysing the diagnosis, treatemnet pattern, symptoms frequency and recovery trends 

~> This layer converts the raw similarity score into dcotor friendly output which supports with its evidence based decision making for the given new patients